# SpeakUp Gemma 4 T4 Fine-tune

This notebook is the deadline-safe Colab T4 path. It uses **Gemma 4 E2B only** with Unsloth QLoRA settings that fit in 16GB VRAM. It saves a small LoRA adapter you can download as proof for Kaggle and optionally merge/export later.

**Do not upload a 5GB GGUF unless you have storage and time.** For the hackathon, the public repo, notebook, training logs, and LoRA adapter artifact are the practical proof. If Kaggle storage is tight, attach the adapter zip or publish it as a Kaggle Model/Hugging Face adapter instead of uploading a full quantized model file.

In [ ]:
%%capture
!pip install -U unsloth
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
!git clone https://github.com/Hetul803/speakup.git
%cd speakup/finetune

In [ ]:
import json
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

MODEL_NAME = 'unsloth/gemma-4-E2B-it-unsloth-bnb-4bit'
MAX_SEQ_LENGTH = 1024
OUTPUT_DIR = './speakup-gemma4-t4-lora'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    use_rslora=True,
)

def format_conversation(example):
    text = ''
    for msg in example['messages']:
        if msg['role'] == 'system':
            text += f"<start_of_turn>user\n[System: {msg['content']}]\n"
        elif msg['role'] == 'user':
            text += f"{msg['content']}<end_of_turn>\n<start_of_turn>model\n"
        elif msg['role'] == 'assistant':
            text += f"{msg['content']}<end_of_turn>\n"
    return {'text': text}

rows = [json.loads(line) for line in open('dataset/aac_training.jsonl')]
dataset = Dataset.from_list(rows).map(format_conversation, remove_columns=list(rows[0].keys()))
print('Training examples:', len(dataset))

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        output_dir=OUTPUT_DIR,
        report_to='none',
        save_strategy='epoch',
    ),
)
trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved adapter:', OUTPUT_DIR)

In [ ]:
FastLanguageModel.for_inference(model)
prompt = "<start_of_turn>user\nCommunicator points at a cup and makes a soft mmm sound. Memory says soft mmm + cup means water. Return SpeakUp JSON.<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=220, temperature=0.2)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
!zip -r speakup-gemma4-t4-lora.zip speakup-gemma4-t4-lora
from google.colab import files
files.download('speakup-gemma4-t4-lora.zip')